In [ ]:
import deepchem as dc
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import scipy
import weightedviews as CP
from weightedviews import load_data, speciesmap
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import mean_squared_error


In [ ]:
tasks, datasets, transformers = dc.molnet.load_sampl(
    splitter=None,
    featurizer="Raw"
)

dataset = datasets[0]

In [ ]:
from smiles_to_mollist import smiles_to_mollist
freesolv_mollist = smiles_to_mollist(dataset.ids)

In [ ]:
y = dataset.y.reshape(-1, 1)

train_mollist, test_mollist, y_train, y_test = train_test_split(
    freesolv_mollist,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [ ]:
freesolv_train = train_mollist
freesolv_test = test_mollist
freesolv_labels_train = y_train
freesolv_labels_test = y_test

ws_freesolv_train, vs_freesolv_train, Natoms, Nviews = load_data(
    freesolv_train,
    setNviews=None,
    carbonbased=False,
    splitx=False,
    verbose=0
)

freesolvG_train = [ws_freesolv_train, vs_freesolv_train]

ws_freesolv_test, vs_freesolv_test, Natoms, Nviews = load_data(
    freesolv_test,
    setNviews=Natoms,
    setNatoms=Natoms,
    carbonbased=False,
    splitx=False,
    verbose=0
)

freesolvG_test = [ws_freesolv_test, vs_freesolv_test]

In [ ]:
labelsG_train = freesolv_labels_train
labelsG_test = freesolv_labels_test

Ntoxicity = 1

ws_train, vs_train = ws_freesolv_train, vs_freesolv_train
ws_test, vs_test = ws_freesolv_test, vs_freesolv_test

dataG_train = [ws_train, vs_train]
dataG_test = [ws_test, vs_test]

print(type(labelsG_train))
print(type(labelsG_test))
print(type(dataG_train))
print(type(dataG_test))
print(ws_train.shape)
print(vs_train.shape)
print(ws_test.shape)
print(vs_test.shape)
print(labelsG_train.shape)
print(labelsG_test.shape)

In [ ]:
def multiDense(Nin, Nout, Nhidden, widthhidden=None, kernel_regularizer=None):
    """Construct a basic NN with some dense layers."""
    if widthhidden is None:
        widthhidden = Nin + Nout

    x = inputs = keras.Input(shape=(Nin,), name='multiDense_input')

    if kernel_regularizer is not None:
        print("Using regularization")

    for i in range(Nhidden):
        x = layers.Dense(
            widthhidden,
            activation='relu',
            kernel_regularizer=kernel_regularizer,
            name='dense' + str(i)
        )(x)

    outputs = layers.Dense(Nout, name='multiDense_output')(x)

    return keras.Model(inputs=inputs, outputs=outputs)


def parallelwrapper(Nparallel, basemodel, insteadmax=False):
    """Construct a model that applies a basemodel multiple times and take a weighted sum."""
    Nin = basemodel.inputs[0].shape[1]
    Nout = basemodel.outputs[0].shape[1]

    parallel_inputs = keras.Input(shape=(Nparallel, Nin), name='parallelwrapper_input0')

    parallel_inputsunstacked = tf.keras.ops.unstack(parallel_inputs, Nparallel, 1)
    xbunstacked = [basemodel(x) for x in parallel_inputsunstacked]
    xb = tf.keras.ops.stack(xbunstacked, axis=1)

    weight_inputs = keras.Input(shape=(Nparallel,), name='parallelScalars')

    if insteadmax:
        out = layers.MaxPool1D(pool_size=Nparallel)(xb)
        out = layers.Reshape((Nout,))(out)
    else:
        out = layers.Dot((-2, -1))([xb, weight_inputs])

    return keras.Model(inputs=[weight_inputs, parallel_inputs], outputs=out, name='parallelwrapper')


from tensorflow.keras import layers, regularizers

def init_generator(data, labels, baselayers, Nfeatures, endlayers,
                   base_regularizer=None, end_regularizer=None):
    """Initialize the generator neural net."""
    insteadmax = False

    Gbase = multiDense(
        data[1].shape[2],
        Nfeatures,
        baselayers,
        kernel_regularizer=base_regularizer
    )

    Gpw = parallelwrapper(Nviews, Gbase, insteadmax)

    Gft = multiDense(
        Nfeatures,
        labels.shape[1],
        endlayers,
        kernel_regularizer=end_regularizer
    )

    generator = keras.Model(
        inputs=Gpw.inputs,
        outputs=Gft(Gpw.outputs),
        name='generator'
    )

    generator.compile(optimizer='adam', loss='mse')

    return generator


baselayers = 3
base_reg = 0.01
Nfeatures = 10
endlayers = 3
end_reg = 0.01

if base_reg == 0:
    base_regularizer = None
else:
    base_regularizer = regularizers.l2(base_reg)

if end_reg == 0:
    end_regularizer = None
else:
    end_regularizer = regularizers.l2(end_reg)

print("(baselayers, base_reg, Nfeatures, endlayers, end_reg) =",
      (baselayers, base_reg, Nfeatures, endlayers, end_reg))

labelsG_train = labelsG_train.reshape(-1, 1)
labelsG_test = labelsG_test.reshape(-1, 1)


In [ ]:
generator = init_generator(
    dataG_train,
    labelsG_train,
    baselayers,
    Nfeatures,
    endlayers,
    base_regularizer=base_regularizer,
    end_regularizer=end_regularizer
)

generator.compile(optimizer='adam', loss='mse')

BATCH_SIZE = 32

if 1:
    history = generator.fit(
        dataG_train,
        labelsG_train,
        epochs=150,
        batch_size=BATCH_SIZE,
        verbose=0,
        validation_data=(dataG_test, labelsG_test)
    )

    print("train loss=", generator.evaluate(dataG_train, labelsG_train, verbose=0))
    print("test loss=", generator.evaluate(dataG_test, labelsG_test, verbose=0))

predicted_train = generator.predict(dataG_train)
predicted_test = generator.predict(dataG_test)

rmse_train = np.sqrt(np.mean((predicted_train - labelsG_train) ** 2))
rmse_test = np.sqrt(np.mean((predicted_test - labelsG_test) ** 2))

print("Train RMSE:", rmse_train)
print("Test RMSE:", rmse_test)

# 2) Coulomb Matrix

In [ ]:
from coulomb_matrix import create_coulomb_matrix

CM_train = create_coulomb_matrix(train_mollist)
CM_test = create_coulomb_matrix(test_mollist, max_atoms=44)

print(CM_train.shape)
print(CM_test.shape)

In [ ]:
CM_train = CM_train.reshape(CM_train.shape[0], -1)
CM_test = CM_test.reshape(CM_test.shape[0], -1)

print(CM_train.shape)
print(CM_test.shape)

In [ ]:
def estimate_width(input_dim,
                   output_dim=1,
                   hidden_layers=3,
                   target_params=479393):
    """
    Estimate hidden-layer width needed to obtain approximately
    target_params trainable parameters.
    """

    low = 1
    high = 5000

    while low <= high:

        width = (low + high) // 2

        params = 0

        # first hidden layer
        params += (input_dim + 1) * width

        # remaining hidden layers
        for _ in range(hidden_layers - 1):
            params += (width + 1) * width

        # output layer
        params += (width + 1) * output_dim

        if params < target_params:
            low = width + 1
        else:
            high = width - 1

    return width


def init_dense_model(
    X,
    y,
    hidden_layers=3,
    target_params=479393,
    regularizer=None,
):

    width = estimate_width(
        input_dim=X.shape[1],
        output_dim=y.shape[1],
        hidden_layers=hidden_layers,
        target_params=target_params,
    )

    print(f"Hidden width selected = {width}")

    inputs = keras.Input(shape=(X.shape[1],))

    x = inputs

    for _ in range(hidden_layers):
        x = layers.Dense(
            width,
            activation="relu",
            kernel_regularizer=regularizer,
        )(x)

    outputs = layers.Dense(y.shape[1])(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="mse",
    )

    print("Total trainable parameters =", model.count_params())

    return model
model = init_dense_model(
    CM_train,
    y_train,
    hidden_layers=3,
    target_params=479393,
    regularizer=regularizers.l2(0.01),
)

In [ ]:
history = model.fit(
    CM_train,
    y_train,
    epochs=150,
    batch_size=32,
    validation_data=(CM_test, y_test),
    verbose=0
)

In [ ]:
print("Train MSE =", model.evaluate(CM_train, y_train, verbose=0))
print("Test MSE =", model.evaluate(CM_test, y_test, verbose=0))

# Predictions
pred_train = model.predict(CM_train, verbose=0)
pred_test = model.predict(CM_test, verbose=0)

# RMSE
rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))

print("Train RMSE:", rmse_train)
print("Test RMSE:", rmse_test)

# 3) Bag of Bonds

In [ ]:
from bag_of_bonds import create_bob
BOB_train, bag_sizes = create_bob(train_mollist)

BOB_test, _ = create_bob(
    test_mollist,
    bag_sizes=bag_sizes
)

print(BOB_train.shape)
print(BOB_test.shape)

In [ ]:
# Build model
model = init_dense_model(
    BOB_train,
    y_train,
    hidden_layers=3,
    target_params=479393,
    regularizer=regularizers.l2(0.01),
)

# Train
history = model.fit(
    BOB_train,
    y_train,
    epochs=150,
    batch_size=32,
    validation_data=(BOB_test, y_test),
    verbose=0,
)

# Evaluate
train_mse = model.evaluate(BOB_train, y_train, verbose=0)
test_mse = model.evaluate(BOB_test, y_test, verbose=0)

print("Train MSE:", train_mse)
print("Test MSE:", test_mse)

# Predictions
pred_train = model.predict(BOB_train, verbose=0)
pred_test = model.predict(BOB_test, verbose=0)

# RMSE
rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))

print("Train RMSE:", rmse_train)
print("Test RMSE:", rmse_test)

# 4) ACSF

In [ ]:
from acsf import create_acsf
ACSF_train, max_atoms = create_acsf(train_mollist)

ACSF_test, _ = create_acsf(
    test_mollist,
    max_atoms=max_atoms
)

print(ACSF_train.shape)
print(ACSF_test.shape)

In [ ]:
ACSF_train = ACSF_train.reshape(ACSF_train.shape[0], -1)
ACSF_test = ACSF_test.reshape(ACSF_test.shape[0], -1)

print(ACSF_train.shape)
print(ACSF_test.shape)

In [ ]:
model = init_dense_model(
    ACSF_train,
    y_train,
    hidden_layers=3,
    target_params=479393,
    regularizer=regularizers.l2(0.01),
)

# Train
history = model.fit(
    ACSF_train,
    y_train,
    epochs=150,
    batch_size=32,
    validation_data=(ACSF_test, y_test),
    verbose=0,
)

# Evaluate
train_mse = model.evaluate(ACSF_train, y_train, verbose=0)
test_mse = model.evaluate(ACSF_test, y_test, verbose=0)

print("Train MSE:", train_mse)
print("Test MSE:", test_mse)

# Predictions
pred_train = model.predict(ACSF_train, verbose=0)
pred_test = model.predict(ACSF_test, verbose=0)

# RMSE
rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))

print("Train RMSE:", rmse_train)
print("Test RMSE:", rmse_test)

# 5) SOAP

In [ ]:
from soap import create_soap
SOAP_train = create_soap(train_mollist)
SOAP_test = create_soap(test_mollist)
print(SOAP_train.shape)
print(SOAP_test.shape)


In [ ]:
model = init_dense_model(
    SOAP_train,
    y_train,
    hidden_layers=3,
    target_params=479393,
    regularizer=regularizers.l2(0.01),
)

# Train
history = model.fit(
    SOAP_train,
    y_train,
    epochs=150,
    batch_size=32,
    validation_data=(SOAP_test, y_test),
    verbose=0,
)

# Evaluate
train_mse = model.evaluate(SOAP_train, y_train, verbose=0)
test_mse = model.evaluate(SOAP_test, y_test, verbose=0)

print("Train MSE:", train_mse)
print("Test MSE:", test_mse)

# Predictions
pred_train = model.predict(SOAP_train, verbose=0)
pred_test = model.predict(SOAP_test, verbose=0)

# RMSE
rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))

print("Train RMSE:", rmse_train)
print("Test RMSE:", rmse_test)